In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#@title JS Shell
%%html
<div id=term_demo></div>
<script src="https://code.jquery.com/jquery-latest.js"></script>
<script src="https://cdn.jsdelivr.net/npm/jquery.terminal/js/jquery.terminal.min.js"></script>
<link href="https://cdn.jsdelivr.net/npm/jquery.terminal/css/jquery.terminal.min.css" rel="stylesheet"/>
<script>
  $('#term_demo').terminal(function(command) {
      if (command !== '') {
          try {
              var result = window.eval(command);
              if (result !== undefined) {
                  this.echo(new String(result));
              }
          } catch(e) {
              this.error(new String(e));
          }
      } else {
          this.echo('');
      }
  }, {
      greetings: 'Welcome to JavaScript Shell',
      name: 'js_demo',
      height: 200,
      prompt: 'js> '
  });

In [3]:
from IPython.display import JSON
from google.colab import output
from subprocess import getoutput
import os

def shell(command):
  if command.startswith('cd'):
    path = command.strip().split(maxsplit=1)[1]
    os.chdir(path)
    return JSON([''])
  return JSON([getoutput(command)])
output.register_callback('shell', shell)

In [4]:
#@title Colab Shell
%%html
<div id=term_demo></div>
<script src="https://code.jquery.com/jquery-latest.js"></script>
<script src="https://cdn.jsdelivr.net/npm/jquery.terminal/js/jquery.terminal.min.js"></script>
<link href="https://cdn.jsdelivr.net/npm/jquery.terminal/css/jquery.terminal.min.css" rel="stylesheet"/>
<script>
  $('#term_demo').terminal(async function(command) {
      if (command !== '') {
          try {
              let res = await google.colab.kernel.invokeFunction('shell', [command])
              let out = res.data['application/json'][0]
              this.echo(new String(out))
          } catch(e) {
              this.error(new String(e));
          }
      } else {
          this.echo('');
      }
  }, {
      greetings: 'Welcome to Colab Shell',
      name: 'colab_demo',
      height: 250,
      prompt: 'colab > '
  });

In [11]:
cd /content/drive/MyDrive/

/content/drive/MyDrive


In [13]:
# Создаём папки retail_dashboard и data
!mkdir -p retail_dashboard/data

In [16]:
ls retail_dashboard/

data/  retail_dasboard.ipynb


In [23]:
!ls /content/drive/MyDrive/retail_dashboard/data

sales_2025_2026.xlsx


In [25]:
import pandas as pd

# Путь к твоему Excel
excel_path = "/content/drive/MyDrive/retail_dashboard/data/sales_2025_2026.xlsx"

# Путь, где сохранить CSV
csv_output = "/content/drive/MyDrive/retail_dashboard/data/retail_aggregated.csv"

# Читаем Excel
xl = pd.ExcelFile(excel_path)

# Список для хранения всех листов
all_sheets = []

print("Обработка листов Excel...")
for sheet_name in xl.sheet_names:
    print(f"  Лист: {sheet_name}")
    df = xl.parse(sheet_name)

    # Добавляем столбец с месяцем (из названия листа)
    df["Месяц"] = sheet_name

    all_sheets.append(df)

# Объединяем все листы в один DataFrame
df_final = pd.concat(all_sheets, ignore_index=True)

# Сохраняем в CSV
df_final.to_csv(csv_output, index=False, encoding="utf-8")
print("✅ retail_aggregated.csv успешно создан!")

# Покажем пример первых строк
print("\nПример первых строк:")
print(df_final.head())

Обработка листов Excel...
  Лист: Январь 25
  Лист: Февраль 25
  Лист: Март 25
  Лист: Апрель 25
  Лист: Май 25
  Лист: Июнь 25
  Лист: Июль 25
  Лист: Август 25
  Лист: Сентябрь 25
  Лист: Октябрь 25
  Лист: Ноябрь 25
  Лист: Декабрь 25
  Лист: Январь 26
  Лист: Февраль 26
  Лист: Март 26
  Лист: Апрель 26
✅ retail_aggregated.csv успешно создан!

Пример первых строк:
          Локация  Размещение  Площадь  Торг площадь Тип ТО Кол-во стеллажей  \
0       Жилой дом         1.0    204.0         149.8  тип 4               56   
1       Жилой дом         2.0    222.0         165.4  тип 2               75   
2             NaN         NaN      NaN           NaN      -                -   
3  Ком.зданиие/ТЦ         2.0    181.2         146.8      -                -   
4     Ком.зданиие         1.0    241.0         186.5  тип 2               76   

  Режим работы        Дата открытия  \
0   9:00-21:00           31.09.2024   
1  10:00-21:00  2024-08-04 00:00:00   
2          NaN                 

In [28]:
%%writefile retail_dashboard/app.py
import streamlit as st
import pandas as pd


# --- Параметры: ПУТЬ к CSV в Google Drive ---
DATA_PATH = "/content/drive/MyDrive/retail_dashboard/data/retail_aggregated.csv"


# --- Загрузка данных ---
@st.cache_data
def load_data():
    try:
        df = pd.read_csv(DATA_PATH, encoding="utf-8")
        return df
    except FileNotFoundError:
        st.error(f"Файл {DATA_PATH} не найден. Проверьте, что он лежит в Google Drive по указанному пути.")
        st.stop()


df = load_data()


# --- Страница ---
st.title("📊 Дашборд розничной сети (ТО)")


# --- Фильтры ---
st.sidebar.header("🔍 Фильтрация")

# Фильтр по ТО
stores = sorted(df["ТО"].dropna().unique().tolist())
selected_store = st.sidebar.selectbox("ТО", ["Все"] + stores)

# Фильтр по Месяцу
months = sorted(df["Месяц"].dropna().unique().tolist())
selected_month = st.sidebar.selectbox("Месяц", ["Все"] + months)

# Фильтр по Размещение (Жилой дом / ТЦ и т.п.)
placements = pd.Series(df["Размещение"].dropna()).unique()
if len(placements) > 0:
    selected_placement = st.sidebar.selectbox("Размещение", ["Все"] + sorted(placements))
else:
    selected_placement = "Все"

# Тип ТО
types = pd.Series(df["Тип ТО"].dropna()).unique()
if len(types) > 0:
    selected_type = st.sidebar.selectbox("Тип ТО", ["Все"] + sorted(types))
else:
    selected_type = "Все"

# Выбор метрики из числовых столбцов
numeric_cols = df.select_dtypes(include="number").columns.tolist()
metric = st.sidebar.selectbox("Метрика", numeric_cols)


# --- Фильтрация данных ---
data = df.copy()

if selected_store != "Все":
    data = data[data["ТО"] == selected_store]

if selected_month != "Все":
    data = data[data["Месяц"] == selected_month]

if selected_placement != "Все":
    data = data[data["Размещение"] == selected_placement]

if selected_type != "Все":
    data = data[data["Тип ТО"] == selected_type]


# Если нет данных — показываем предупреждение
if len(data) == 0:
    st.info("По выбранным фильтрам нет данных 😓")
else:
    # --- KPI‑карточки ---
    st.subheader("📈 Основные показатели")

    col1, col2, col3 = st.columns(3)
    col1.metric("Среднее значение", f"{data[metric].mean():,.2f}")
    col2.metric("Максимум", f"{data[metric].max():,.2f}")
    col3.metric("Минимум", f"{data[metric].min():,.2f}")


    # --- График по ТО ---
    st.subheader(f"{metric} по ТО ({selected_store if selected_store != 'Все' else 'все'})")

    if len(data["ТО"].unique()) == 1:
        st.write("Данные только для одного ТО.")
    else:
        chart_data = data.set_index("ТО")[metric]
        st.bar_chart(chart_data)


    # --- График динамики по месяцам ---
    st.subheader(f"📊 Динамика {metric} по месяцам")

    all_months_data = (
        df.groupby(["Месяц"], as_index=False)[metric]
        .sum()
        .set_index("Месяц")
    )
    st.line_chart(all_months_data)


    # --- Таблица с сырыми данными ---
    st.subheader("📋 Сырые данные (ТО и метрики)")
    st.dataframe(
        data[["ТО", "Размещение", "Тип ТО", "Месяц", metric]],
        height=400
    )

Writing app.py


In [18]:
%%writefile retail_dashboard/requirements.txt
streamlit>=1.30
pandas>=2.0
openpyxl
sqlalchemy

Writing retail_dashboard/requirements.txt


In [19]:
!zip -r retail_dashboard.zip retail_dashboard/

  adding: retail_dashboard/ (stored 0%)
  adding: retail_dashboard/retail_dasboard.ipynb (deflated 73%)
  adding: retail_dashboard/data/ (stored 0%)
  adding: retail_dashboard/app.py (deflated 57%)
  adding: retail_dashboard/requirements.txt (stored 0%)


In [20]:
from google.colab import files
files.download("retail_dashboard.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>